In [6]:
import pandas as pd
import random
import os
import numpy as np

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


abcde 각각 해석해서 이름 붙이기

In [8]:
#parquet -> 대용량 데이터셋을 다루기 위한 확장자
#csv -> 24만권 => 1. 신용정보, ... , 8. 8개의 정보 프래임을 열은 다 살리고 중복 열만 죽임
#열 갯수 -> 700개

In [9]:
train_path = '/content/drive/MyDrive/블루문/card_train.csv'
test_path = '/content/drive/MyDrive/블루문/card_test.csv'

In [10]:
train_df = pd.read_csv(train_path)

In [11]:
train_df .shape

(70560, 738)

In [12]:
train_df.head()

,Unnamed: 0.1,대표결제일,대표결제방법코드,대표청구지고객주소구분코드,대표청구서수령지구분코드,청구서수령방법,청구서발송여부_B0,청구서발송여부_R3M,청구서발송여부_R6M,청구금액_B0,...,컨택건수_CA_청구서_R6M,컨택건수_이용유도_청구서_R6M,컨택건수_이용유도_인터넷_R6M,컨택건수_이용유도_당사앱_R6M,컨택건수_채권_B0M,컨택건수_채권_R6M,캠페인접촉건수_R12M,캠페인접촉일수_R12M,ID,Segment.1
0,61865,1,자동이체,주거지,우편,우편,1,1,1,466,...,0,5,5,0,0,0,1회 이상,1일 이상,TRAIN_000389,E
1,8547,13,자동이체,미확인,이메일,이메일,1,1,1,2417,...,0,4,6,0,0,0,1회 이상,1일 이상,TRAIN_304152,D
2,43497,25,자동이체,주거지,우편,우편,0,0,0,0,...,2,4,2,0,0,0,5회 이상,5일 이상,TRAIN_356239,E
3,40785,25,자동이체,주거지,우편,우편,1,1,1,5931,...,0,0,0,0,0,0,1회 이상,1일 이상,TRAIN_122432,E
4,18999,25,자동이체,주거지,우편,우편,1,1,1,684,...,2,0,0,0,0,0,10회 이상,10일 이상,TRAIN_285200,E


In [13]:
for i in train_df.columns:
  print(i)

Unnamed: 0.1
대표결제일
대표결제방법코드
대표청구지고객주소구분코드
대표청구서수령지구분코드
청구서수령방법
청구서발송여부_B0
청구서발송여부_R3M
청구서발송여부_R6M
청구금액_B0
청구금액_R3M
청구금액_R6M
포인트_마일리지_건별_B0M
포인트_마일리지_건별_R3M
포인트_포인트_건별_B0M
포인트_포인트_건별_R3M
포인트_마일리지_월적립_B0M
포인트_마일리지_월적립_R3M
포인트_포인트_월적립_B0M
포인트_포인트_월적립_R3M
포인트_적립포인트_R12M
포인트_적립포인트_R3M
포인트_이용포인트_R12M
포인트_이용포인트_R3M
포인트_잔여포인트_B0M
마일_적립포인트_R12M
마일_적립포인트_R3M
마일_이용포인트_R12M
마일_이용포인트_R3M
마일_잔여포인트_B0M
할인건수_R3M
할인금액_R3M
할인건수_B0M
할인금액_B0M
할인금액_청구서_R3M
할인금액_청구서_B0M
상환개월수_결제일_R6M
상환개월수_결제일_R3M
선결제건수_R6M
선결제건수_R3M
연체건수_R6M
연체건수_R3M
혜택수혜금액_R3M
포인트_마일리지_환산_B0M
혜택수혜금액
최초한도금액
카드이용한도금액
CA한도금액
일시상환론한도금액
월상환론한도금액
CA이자율_할인전
CL이자율_할인전
RV일시불이자율_할인전
RV현금서비스이자율_할인전
RV신청일자
RV약정청구율
RV최소결제비율
자발한도감액횟수_R12M
자발한도감액금액_R12M
자발한도감액후경과월
강제한도감액횟수_R12M
강제한도감액금액_R12M
강제한도감액후경과월
한도증액횟수_R12M
한도증액금액_R12M
한도증액후경과월
상향가능한도금액
상향가능CA한도금액
카드론동의여부
월상환론상향가능한도금액
RV전환가능여부
일시불ONLY전환가능여부
카드이용한도금액_B1M
카드이용한도금액_B2M
특별한도보유여부_R3M
연체감액여부_R3M
한도심사요청건수
한도요청거절건수
한도심사요청후경과월
한도심사거절후경과월
시장단기연체여부_R6M
rv최초시작후경과일
증감율_이용건수_신용_전월
증감율_이용건수_신판_전월
증감율_이용건수_일시불_전

In [14]:
X = train_df.drop(columns=['ID', 'Unnamed: 0.1', 'Segment.1', 'Segment'])
Y = train_df['Segment']

1. 738개의 열을 어떻게 뽑아내서 붙일 것인가?

In [15]:
#팀에서 시계열 데이터를 다룰 수 있다면 시계열 데이터로 보기
#교수님은 이 데이터를 시계열로 보지 않고 하나의 데이터로 봄
for i in train_df.columns:
  print(i)

Unnamed: 0.1
대표결제일
대표결제방법코드
대표청구지고객주소구분코드
대표청구서수령지구분코드
청구서수령방법
청구서발송여부_B0
청구서발송여부_R3M
청구서발송여부_R6M
청구금액_B0
청구금액_R3M
청구금액_R6M
포인트_마일리지_건별_B0M
포인트_마일리지_건별_R3M
포인트_포인트_건별_B0M
포인트_포인트_건별_R3M
포인트_마일리지_월적립_B0M
포인트_마일리지_월적립_R3M
포인트_포인트_월적립_B0M
포인트_포인트_월적립_R3M
포인트_적립포인트_R12M
포인트_적립포인트_R3M
포인트_이용포인트_R12M
포인트_이용포인트_R3M
포인트_잔여포인트_B0M
마일_적립포인트_R12M
마일_적립포인트_R3M
마일_이용포인트_R12M
마일_이용포인트_R3M
마일_잔여포인트_B0M
할인건수_R3M
할인금액_R3M
할인건수_B0M
할인금액_B0M
할인금액_청구서_R3M
할인금액_청구서_B0M
상환개월수_결제일_R6M
상환개월수_결제일_R3M
선결제건수_R6M
선결제건수_R3M
연체건수_R6M
연체건수_R3M
혜택수혜금액_R3M
포인트_마일리지_환산_B0M
혜택수혜금액
최초한도금액
카드이용한도금액
CA한도금액
일시상환론한도금액
월상환론한도금액
CA이자율_할인전
CL이자율_할인전
RV일시불이자율_할인전
RV현금서비스이자율_할인전
RV신청일자
RV약정청구율
RV최소결제비율
자발한도감액횟수_R12M
자발한도감액금액_R12M
자발한도감액후경과월
강제한도감액횟수_R12M
강제한도감액금액_R12M
강제한도감액후경과월
한도증액횟수_R12M
한도증액금액_R12M
한도증액후경과월
상향가능한도금액
상향가능CA한도금액
카드론동의여부
월상환론상향가능한도금액
RV전환가능여부
일시불ONLY전환가능여부
카드이용한도금액_B1M
카드이용한도금액_B2M
특별한도보유여부_R3M
연체감액여부_R3M
한도심사요청건수
한도요청거절건수
한도심사요청후경과월
한도심사거절후경과월
시장단기연체여부_R6M
rv최초시작후경과일
증감율_이용건수_신용_전월
증감율_이용건수_신판_전월
증감율_이용건수_일시불_전

In [16]:
recent3 = [m for m in X.columns if 'R3M' in m]
print(recent3)
x_train = X[recent3]

['청구서발송여부_R3M', '청구금액_R3M', '포인트_마일리지_건별_R3M', '포인트_포인트_건별_R3M', '포인트_마일리지_월적립_R3M', '포인트_포인트_월적립_R3M', '포인트_적립포인트_R3M', '포인트_이용포인트_R3M', '마일_적립포인트_R3M', '마일_이용포인트_R3M', '할인건수_R3M', '할인금액_R3M', '할인금액_청구서_R3M', '상환개월수_결제일_R3M', '선결제건수_R3M', '연체건수_R3M', '혜택수혜금액_R3M', '특별한도보유여부_R3M', '연체감액여부_R3M', '혜택수혜율_R3M', '이용금액_R3M_신용체크', '이용금액_R3M_신용', '이용금액_R3M_신용_가족', '이용금액_R3M_체크', '홈페이지_금융건수_R3M', '홈페이지_선결제건수_R3M', 'RV_평균잔액_R3M', 'RV_최대잔액_R3M', '이용건수_신용_R3M', '이용건수_신판_R3M', '이용건수_일시불_R3M', '이용건수_할부_R3M', '이용건수_할부_유이자_R3M', '이용건수_할부_무이자_R3M', '이용건수_부분무이자_R3M', '이용건수_CA_R3M', '이용건수_체크_R3M', '이용건수_카드론_R3M', '이용금액_일시불_R3M', '이용금액_할부_R3M', '이용금액_할부_유이자_R3M', '이용금액_할부_무이자_R3M', '이용금액_부분무이자_R3M', '이용금액_CA_R3M', '이용금액_체크_R3M', '이용금액_카드론_R3M', '이용개월수_신용_R3M', '이용개월수_신판_R3M', '이용개월수_일시불_R3M', '이용개월수_할부_R3M', '이용개월수_할부_유이자_R3M', '이용개월수_할부_무이자_R3M', '이용개월수_부분무이자_R3M', '이용개월수_CA_R3M', '이용개월수_체크_R3M', '이용개월수_카드론_R3M', '이용금액_온라인_R3M', '이용금액_오프라인_R3M', '이용건수_온라인_R3M', '이용건수_오프라인_R3M', '이용금액_페이_온라인_R3M', '이용금액_페

In [17]:
x_train.head()

,청구서발송여부_R3M,청구금액_R3M,포인트_마일리지_건별_R3M,포인트_포인트_건별_R3M,포인트_마일리지_월적립_R3M,포인트_포인트_월적립_R3M,포인트_적립포인트_R3M,포인트_이용포인트_R3M,마일_적립포인트_R3M,마일_이용포인트_R3M,...,이용금액_연체_R3M,이용개월수_전체_R3M,이용개월수_결제일_R3M,건수_할부전환_R3M,금액_할부전환_R3M,승인거절건수_R3M,승인거절건수_한도초과_R3M,승인거절건수_BL_R3M,승인거절건수_입력오류_R3M,승인거절건수_기타_R3M
0,1,1678,0,0,0,0,0,0,0,0,...,0,3,3,0,0,0,0,0,0,0
1,1,7778,0,0,0,0,0,0,0,0,...,0,3,3,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,15957,0,3743,0,0,3743,2285,0,0,...,0,3,3,0,0,0,0,0,0,0
4,1,2661,0,0,0,0,0,0,0,0,...,0,3,3,0,0,0,0,0,0,0


In [18]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70560 entries, 0 to 70559
Data columns (total 88 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   청구서발송여부_R3M       70560 non-null  int64  
 1   청구금액_R3M          70560 non-null  int64  
 2   포인트_마일리지_건별_R3M   70560 non-null  int64  
 3   포인트_포인트_건별_R3M    70560 non-null  int64  
 4   포인트_마일리지_월적립_R3M  70560 non-null  int64  
 5   포인트_포인트_월적립_R3M   70560 non-null  int64  
 6   포인트_적립포인트_R3M     70560 non-null  int64  
 7   포인트_이용포인트_R3M     70560 non-null  int64  
 8   마일_적립포인트_R3M      70560 non-null  int64  
 9   마일_이용포인트_R3M      70560 non-null  int64  
 10  할인건수_R3M          70560 non-null  object 
 11  할인금액_R3M          70560 non-null  int64  
 12  할인금액_청구서_R3M      70560 non-null  int64  
 13  상환개월수_결제일_R3M     70560 non-null  int64  
 14  선결제건수_R3M         70560 non-null  int64  
 15  연체건수_R3M          70560 non-null  int64  
 16  혜택수혜금액_R3M        70560 non-null  int64 

In [19]:
name = [x for x in x_train.columns if x_train[x].dtype == 'object']

In [20]:
name

['할인건수_R3M']

In [21]:
type(x_train)

pandas.core.frame.DataFrame

In [22]:
x_train['할인건수_R3M'].unique()

array(['1회 이상', '10회 이상', '20회 이상', '30회 이상'], dtype=object)

In [23]:
#라벨 인코딩을 하면 됨
#열을 구성해 오면 멘토링 시간에 봐줌

1. 모든 열을 출력해 본다
2. 최근 3개월 r3m만 추출해서 분류해 보자
3. 숫자가 아닌 것 object
4. 데이터 안에 카테고라가 몇 개인가 ['할인건수']
5. 할인건수를 잘 인코딩해서 열에 붙여 넣는다

In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
x_train['할인건수_R3M'] = le.fit_transform(x_train['할인건수_R3M'])

<ipython-input-24-4dc9050408e3>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x_train['할인건수_R3M'] = le.fit_transform(x_train['할인건수_R3M'])


In [25]:
#DT, RF -> 계열이 과적합
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf = rf.fit(x_train, Y)

2. 클래스 불균형

In [26]:
Y.value_counts()

,count
Segment,
E,56505
D,10270
C,3753
A,28
B,4
